# Ingest drivers.json file
1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
   - Source File
   - Ingestion Timestamp
3. Write to bronze delta table  

In [0]:
%run "../00-common/01.environment-config"


In [0]:
%run "../00-common/02.bronze-helpers"

In [0]:
val source_file= landing_folder_path + "/drivers.json"
val table_name= catalog_name + "." + bronze_schema + "." + "drivers"

In [0]:
import org.apache.spark.sql.types.{StructType,StructField,StringType,DateType}

val name_schema = StructType(Seq(
    StructField("givenName", StringType),
    StructField("familyName", StringType)
))

val drivers_schema=StructType(Seq(
  StructField("driverId", StringType),
  StructField("name", name_schema),
  StructField("dateOfBirth", DateType),
  StructField("nationality", StringType),
    StructField("url", StringType),
))

val drivers_df=spark.read.format("json")
.option("mode", "FAILFAST")
.schema(drivers_schema)
.load(source_file)

display(drivers_df)

In [0]:

val drivers_final_df= add_ingestion_metadata(drivers_df)

display(drivers_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
drivers_final_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
display(spark.table(table_name))